<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "132"
iso_country = "Cabo Verde"
year = 2023
import_portfolio = "h5_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h5_ag6_by_sea_X.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 2.500*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Cabo Verde': 

Number of trade records present in dataset: 14218
Number of partner countries present in dataset: 107
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$1617.3m
Total weight within the dataset: 2396.6 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,132,Cabo Verde,620,Portugal,Import,Sea,291211,"291211 | Aldehydes; acyclic,",2912,"2912 | Aldehydes, whether o",29,29 | Organic chemicals,3417,"Ethers, alcohol peroxides, ether peroxides, ep...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",3113.898,2860.660,3113.898,kg,1283.0,1283.0,1431.147521,2575.0,1600.030325,2289.879433,['Container'],['Container'],11.968809,11.947911,11.949953,11.949953,11.959318,11.959318,12.519982,22.189337,13.809942,12.821246,12.856066,13.178057,12.259262,21.731058,14.535835,13.314607,13.009064,13.598245,12.030597,23.631052,18.025133,15.764145,14.584818,14.173398,11.806168,25.125600,20.675483,19.374682,15.614925,15.190336,11.585976,25.461276,24.325996,24.535153,16.928201,15.956958,11.361370,24.937946,23.898193,26.183982,19.368111,16.249425,11.149776,22.245253,23.185475,24.295958,18.213018,16.441420,10.934016,20.434166,20.771864,25.910220,21.405477,16.550296,3417
1,1,132,Cabo Verde,724,Spain,Import,Sea,291211,"291211 | Aldehydes; acyclic,",2912,"2912 | Aldehydes, whether o",29,29 | Organic chemicals,3417,"Ethers, alcohol peroxides, ether peroxides, ep...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",20.987,15.682,20.987,kg,10.0,10.0,9.645625,2711.0,1684.536781,16.248410,['Container'],['Container'],0.084928,0.084779,0.084794,0.084794,0.084860,0.084860,0.088839,0.157450,0.097992,0.090976,0.091223,0.093508,0.086989,0.154198,0.103143,0.094477,0.092309,0.096490,0.085366,0.167680,0.127902,0.111858,0.103490,0.100571,0.083774,0.178285,0.146708,0.137478,0.110800,0.107787,0.082211,0.180667,0.172611,0.174095,0.120118,0.113227,0.080617,0.176953,0.169576,0.185795,0.137431,0.115302,0.079116,0.157847,0.164518,0.172398,0.129235,0.116664,0.077585,0.144996,0.147392,0.183852,0.151888,0.117437,3417


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $343.0m (21.2%), ... 

                             cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                            
10 | Cereals               48494606.9  132682682.9      3.0      5.5
04 | Dairy produce; birds  41704068.2   23726802.3      2.6      1.0
02 | Meat and edible meat  31956804.2   14106293.8      2.0      0.6
15 | Animal, vegetable or  30401257.2   19253273.6      1.9      0.8
03 | Fish and crustaceans  28428032.5    8344784.1      1.8      0.3
22 | Beverages, spirits a  27864046.3   61171727.6      1.7      2.6
20 | Preparations of vege  19805867.9   14104247.1      1.2      0.6
07 | Vegetables and certa  17992485.9   29477820.7      1.1      1.2
08 | Fruit and nuts, edib  17503355.1   16302060.4      1.1      0.7
19 | Preparations of cere  17039214.3    6395600.0      1.1      0.3
21 | Miscellaneous edible  15424375.8    4446987.7      1.0      0.2
16

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $305.5m (18.9%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       184872585.8  216372354.3     11.4      9.0
Food & Beverages  109957870.6  119818142.7      6.8      5.0
Fishing            10636023.5    4556466.0      0.7      0.2



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $280.4m (17.3%), ... 

                                 cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
OFD - Food products n.e.c.     76547479.7  43585144.4      4.7      1.8
MIL - Dairy products           40096292.5  21820860.9      2.5      0.9
OMT - Meat products n.e.c.     38288650.5  15303874.2      2.4      0.6
VOL - Vegetable oils and fats  33608238.6  27782489.0      2.1      1.2
V_F - Vegetables, fruit, nuts  29698278.6  40310548.6      1.8      1.7
PCR - Processed rice           26768556.3  46496982.3      1.7      1.9
GRO - Cereal grains n.e.c.     14807251.0  61139218.3      0.9      2.6
SGR - Sugar                    11148394.2  24545763.2      0.7      1.0
CMT - Bovine meat prods         6262483.9   1652100.3      0.4      0.1
OCR - Crops n.e.c.              2802093.8   1016430.2      0.2      0.0
OAP - Animal products n.e.c.     251392.8     77158.3      0.0      0.

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 21.2% 
GTAP 17.3% 
EORA 18.9%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46347/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,132,Cabo Verde,620,Portugal,Import,Sea,291211,"291211 | Aldehydes; acyclic,",2912,"2912 | Aldehydes, whether o",29,29 | Organic chemicals,3417,"Ethers, alcohol peroxides, ether peroxides, ep...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",3113.898,2860.660,3113.898,kg,1283.0,1283.0,1431.147521,2575.0,1600.030325,2289.879433,['Container'],['Container'],11.968809,11.947911,11.949953,11.949953,11.959318,11.959318,12.519982,22.189337,13.809942,12.821246,12.856066,13.178057,12.259262,21.731058,14.535835,13.314607,13.009064,13.598245,12.030597,23.631052,18.025133,15.764145,14.584818,14.173398,11.806168,25.125600,20.675483,19.374682,15.614925,15.190336,11.585976,25.461276,24.325996,24.535153,16.928201,15.956958,11.361370,24.937946,23.898193,26.183982,19.368111,16.249425,11.149776,22.245253,23.185475,24.295958,18.213018,16.441420,10.934016,20.434166,20.771864,25.910220,21.405477,16.550296,3417,NaN
1,1,132,Cabo Verde,724,Spain,Import,Sea,291211,"291211 | Aldehydes; acyclic,",2912,"2912 | Aldehydes, whether o",29,29 | Organic chemicals,3417,"Ethers, alcohol peroxides, ether peroxides, ep...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",20.987,15.682,20.987,kg,10.0,10.0,9.645625,2711.0,1684.536781,16.248410,['Container'],['Container'],0.084928,0.084779,0.084794,0.084794,0.084860,0.084860,0.088839,0.157450,0.097992,0.090976,0.091223,0.093508,0.086989,0.154198,0.103143,0.094477,0.092309,0.096490,0.085366,0.167680,0.127902,0.111858,0.103490,0.100571,0.083774,0.178285,0.146708,0.137478,0.110800,0.107787,0.082211,0.180667,0.172611,0.174095,0.120118,0.113227,0.080617,0.176953,0.169576,0.185795,0.137431,0.115302,0.079116,0.157847,0.164518,0.172398,0.129235,0.116664,0.077585,0.144996,0.147392,0.183852,0.151888,0.117437,3417,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $241.8m of $1617.3m (15.0%)
Volume: 285.0 of 2396.6 million kilograms (11.9%)
Value-to-Weight: 0.85 from 0.67 US$ per kg (+25.7%)
Average distance to market (volume-weighted): 3874.4 from 3626.5 nm (+6.8%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    404  70953802.0  69311793.2      4.4      2.9
1                Dairy     85  40836396.2  23374454.6      2.5      1.0
2        Fats and Oils     77  29153947.7  17820781.4      1.8      0.7
3        Meat and Prep     73  24480562.1  12311821.2      1.5      0.5
4           Vegetables    260  24306674.4  34510417.5      1.5      1.4
5  Unprocessed Cereals     23  21721025.7  86176972.3      1.3      3.6
6               Fruits    201  17149250.5  16143558.8      1.1      0.7
7      Sugar and Honey     78  13219709.7  25380800.4      0.8      1.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,24698441.3,43525949.7,27181770.9,25263914.6,25331880.8,25975186.0,22878783.8,50014819.9,47764224.7,48249325.3,33313566.3,31384743.6,21608032.9,40196651.4,40859083.1,50754509.0,41940596.7,32537864.0
1,Fats and Oils,16152934.2,27637848.5,17553323.6,16378737.3,16424125.4,16911159.3,15036565.0,31460817.9,30269141.6,30940595.4,21389957.3,20193639.2,14256987.3,25662015.1,26110116.8,31722603.7,26153809.9,20889241.0
2,Fruits,15977109.7,28125353.8,17574480.1,16347085.1,16389121.4,16801621.4,14802047.6,32303655.7,30809157.5,31113975.8,21517084.8,20299001.0,13981587.1,26089296.8,26500568.7,32794310.2,27022963.2,21049121.1
3,Meat and Prep,18690805.6,33125971.5,20616557.6,19140555.3,19192536.7,19673230.0,17296448.5,38010576.7,36315742.4,36627988.1,25271737.3,23821791.2,16323151.5,30505715.7,31009858.1,38680795.8,31955764.8,24707572.4
4,Processed Cereals,220907445.6,390931306.9,243524378.4,226150176.9,226764299.0,232467746.2,204481890.8,448768788.8,428740590.6,432576753.6,298551487.7,281317676.2,193015907.9,360284596.4,366242413.2,456351440.6,377085537.9,291741250.0
5,Sugar and Honey,28902516.5,46335893.1,30689473.0,28963831.6,29051341.3,29994803.4,27200828.0,54949675.3,52488673.4,54067110.0,38243759.3,34839574.3,26012803.1,45002341.4,45823830.0,53152683.9,44804732.4,35804565.4
6,Unprocessed Cereals,143059824.2,237118470.0,153799607.2,144369059.8,144792220.7,149144746.2,133915022.5,278125127.4,265698088.2,271741922.1,190682630.8,175611867.0,127531683.4,226216735.4,230222903.0,273720567.1,229184966.4,181027233.2
7,Vegetables,40890714.4,71926031.3,44964384.6,41807992.9,41920263.6,42991039.2,37891724.7,82690470.9,78959703.9,79800921.8,55102952.7,51905327.0,35794991.9,66481111.1,67581394.1,83807684.6,69259901.8,53804511.4


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.46,0.06,0.01,0.02,0.03,0.66,0.61,0.62,0.26,0.21,0.46,0.47,0.71,0.50,0.27
1,Fats and Oils,0.39,0.05,0.01,0.01,0.03,0.56,0.52,0.55,0.22,0.18,0.39,0.41,0.60,0.41,0.23
2,Fruits,0.71,0.09,0.02,0.02,0.05,1.02,0.93,0.95,0.39,0.32,0.71,0.73,1.10,0.76,0.41
3,Meat and Prep,0.59,0.08,0.02,0.02,0.04,0.85,0.78,0.79,0.33,0.27,0.58,0.60,0.91,0.64,0.34
4,Processed Cereals,2.40,0.32,0.07,0.08,0.16,3.44,3.16,3.21,1.33,1.08,2.36,2.44,3.71,2.59,1.39
5,Sugar and Honey,1.32,0.14,0.00,0.01,0.08,2.10,1.91,2.03,0.84,0.58,1.44,1.50,2.05,1.42,0.74
6,Unprocessed Cereals,4.33,0.49,0.06,0.08,0.28,6.64,6.07,6.35,2.61,1.92,4.54,4.73,6.73,4.68,2.46
7,Vegetables,1.28,0.17,0.04,0.04,0.09,1.84,1.69,1.72,0.71,0.58,1.26,1.31,1.98,1.38,0.74
